In [1]:
import sys
import os

# Get the absolute path to the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

print(f"✅ Added {project_root} to sys.path")

✅ Added c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning to sys.path


In [2]:
# Environment Setup

import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from src.oracle_engine import OracleEngine
from src.symbolic_solver import SymbolicSolver
import config

# Initialize for Manhattan (our densest test case)
config.CURRENT_CITY = "manhattan"
oracle = OracleEngine(config.get_graph_path(), config.get_poi_path())
solver = SymbolicSolver(oracle)

print(f"✅ Oracle loaded with {len(oracle.poi_df)} POIs")
print(f"✅ SCC Map contains {len(solver.scc_lookup)} nodes")

c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\src\oracle_engine.py:23: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  self.G = pickle.load(f)
c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\src\oracle_engine.py:23: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  self.G = pickle.load(f)


📍 Extracting coordinates from manhattan 'centroid' column...
✅ Solver Initialized: Found 1 isolated graph components.
✅ Oracle loaded with 20979 POIs
✅ SCC Map contains 74137 nodes


In [3]:
import pandas as pd
import config

def patched_resolve_all_candidates(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    """Temporary fix for the KeyError: 'semantic_score'"""
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        filtered_df = self.poi_df[
            (self.poi_df['y'] >= lat_min) & (self.poi_df['y'] <= lat_max) &
            (self.poi_df['x'] >= lon_min) & (self.poi_df['x'] <= lon_max)
        ].copy()
    else:
        filtered_df = self.poi_df.copy()

    if filtered_df.empty: return []

    # 1. Initialize the column immediately to prevent KeyError
    filtered_df['semantic_score'] = 1.0

    # 2. Category Filter
    mask = pd.Series(True, index=filtered_df.index)
    for key, val in tags.items():
        if key in filtered_df.columns:
            if val == "yes":
                mask &= filtered_df[key].notna() & (filtered_df[key] != "")
            elif isinstance(val, list):
                mask &= filtered_df[key].isin(val)
            else:
                mask &= (filtered_df[key] == val)
    
    filtered_df = filtered_df[mask].copy()
    if filtered_df.empty: return []

    # 3. Semantic Scoring
    if landmark_name:
        query = landmark_name.lower().replace(" ", "_")
        name_hits = filtered_df['name'].astype(str).str.lower().str.contains(query.replace("_", " "), na=False)
        tag_hits = pd.Series(False, index=filtered_df.index)
        
        for col in config.POI_SEARCH_COLUMNS:
            if col in filtered_df.columns:
                tag_hits |= filtered_df[col].astype(str).str.lower().str.contains(query, na=False)
        
        filtered_df['semantic_score'] = (name_hits.astype(float) * 2.0) + (tag_hits.astype(float) * 1.5)

    # 4. Filter & Map
    candidates_df = filtered_df[filtered_df['semantic_score'] >= score_threshold]
    results = []
    for _, row in candidates_df.iterrows():
        raw_id = str(row['osmid']).replace('#', '')
        node_id = f"{self.prefix}{raw_id}"
        if node_id in self.G:
            results.append({
                "node_id": node_id, 
                "name": row.get('name', 'Unknown'), 
                "score": row['semantic_score'], 
                "coords": (row['y'], row['x'])
            })
    return results

# THE MAGIC LINE: Overwrite the method on the LIVE object
# Replace 'oracle' with whatever your OracleEngine variable name is
oracle.resolve_all_candidates = patched_resolve_all_candidates.__get__(oracle, oracle.__class__)

print("✅ Oracle method patched in memory. Try running your test again!")

✅ Oracle method patched in memory. Try running your test again!


In [4]:
# Testing the "Geodesic Gatekeeper" (Spatial Precision)

import math

# Pick a central point (e.g., near Empire State Building)
center_lat, center_lon = 40.7484, -73.9857 
radius = 1500

# 1. Manually trigger the buffer logic from your resolve_nearby_candidates
lat_buffer = radius / 111000.0
lon_buffer = radius / (111000.0 * math.cos(math.radians(center_lat)))

print(f"Lat Buffer: {lat_buffer:.6f} degrees")
print(f"Lon Buffer: {lon_buffer:.6f} degrees (Curvature Adjusted)")

# 2. Check the "Waterfall" Pruning
bounds = (center_lat - lat_buffer, center_lat + lat_buffer, 
          center_lon - lon_buffer, center_lon + lon_buffer)

candidates = oracle.resolve_all_candidates(tags={'amenity': 'cafe'}, bounds=bounds)
print(f"Found {len(candidates)} cafes within the 1.5km 'Box'")

# 3. Precise Geodesic Verification
nearby = oracle.resolve_nearby_candidates({'amenity': 'cafe'}, center_lat, center_lon)
print(f"Found {len(nearby)} cafes within the precise 1.5km 'Circle'")

Lat Buffer: 0.013514 degrees
Lon Buffer: 0.017838 degrees (Curvature Adjusted)
Found 289 cafes within the 1.5km 'Box'
Found 251 cafes within the precise 1.5km 'Circle'


In [5]:
# The "Solve" Method Stress Test (Ambiguity Logic)

# We need a start node from the graph
sample_start_node = list(oracle.G.nodes())[500]

test_cases = [
    {"label": "Standard Unique", "text": "the post office"},
    {"label": "Broad Category (Ambiguous)", "text": "a deli"},
    {"label": "Underspecified Mask", "text": "the [MASK]"},
    {"label": "Non-existent (Contradictory)", "text": "the space shuttle launchpad"}
]

for case in test_cases:
    print(f"\n🔍 Testing: {case['label']} ('{case['text']}')")
    # Note: We might need to adjust the extraction logic to handle '[MASK]'
    result = solver.solve(case['text'], sample_start_node)
    
    print(f"State: {result['state']}")
    print(f"Candidates Found: {result.get('candidate_count', 0)}")
    if 'reason' in result:
        print(f"Reason: {result['reason']}")


🔍 Testing: Standard Unique ('the post office')
State: Ambiguous
Candidates Found: 10

🔍 Testing: Broad Category (Ambiguous) ('a deli')
State: Ambiguous
Candidates Found: 76

🔍 Testing: Underspecified Mask ('the [MASK]')
State: Ambiguous
Candidates Found: 4251

🔍 Testing: Non-existent (Contradictory) ('the space shuttle launchpad')
State: Ambiguous
Candidates Found: 4251


In [6]:
# SCC Reachability O(1) Performance

import time

start_node = list(oracle.G.nodes())[0]
end_node = list(oracle.G.nodes())[-1]

# Method A: Traditional NetworkX (The "Freeze" risk)
start_a = time.time()
try:
    has_path_nx = nx.has_path(oracle.G, start_node, end_node)
except:
    has_path_nx = "Error"
end_a = time.time()

# Method B: Our New SCC Utility
start_b = time.time()
has_path_scc = solver.check_reachability_scc(start_node, end_node)
end_b = time.time()

print(f"NX Time: {(end_a - start_a)*1000:.2f}ms")
print(f"SCC Time: {(end_b - start_b)*1000:.2f}ms")
print(f"Results Match: {has_path_nx == has_path_scc}")

NX Time: 10.64ms
SCC Time: 0.17ms
Results Match: True


In [7]:
# Testing Specific Identity Resolution
print("🔍 Testing: Named Entity Resolution")
tags = config.LANDMARK_GROUPS["BIKE"]
name = "East Village bicycle shop"
candidates = oracle.resolve_all_candidates(tags=tags, landmark_name=name)

# LOGIC: Because the name is highly specific, candidates should be 1.
state = "Answerable" if len(candidates) == 1 else "Ambiguous"
print(f"State: {state} | Candidates: {len(candidates)}")

🔍 Testing: Named Entity Resolution
State: Ambiguous | Candidates: 0


In [8]:
# Diagnostic: See what BIKE shops actually exist in the POI data
bike_pois = oracle.poi_df[oracle.poi_df['amenity'].isin(['bicycle_parking', 'bicycle_rental']) | 
                          (oracle.poi_df['shop'] == 'bicycle')]

print(f"Total Bike-related POIs: {len(bike_pois)}")
print("Sample names in our data:")
print(bike_pois['name'].dropna().unique()[:10])

Total Bike-related POIs: 2976
Sample names in our data:
['Franks Bike Shop' 'Citi Bike' 'Citi Bike - E 10 St & Avenue A'
 'Citi Bike - St Marks Pl & 1 Ave' 'Citi Bike - E 7 St & Avenue A'
 'Citi Bike - Allen St & Rivington St' 'Citi Bike - Canal St & Seward Pk'
 'Citi Bike - 11 Ave / W 59 St' 'Echelon Cycles'
 'Citi Bike - E 31 St & 3 Ave']


In [9]:
print("🔍 Testing: Exact Named Entity Resolution")
# 1. Use the category from config
tags = config.LANDMARK_GROUPS["BIKE"]

# 2. Use a name we KNOW exists in the sample list
specific_name = "Franks Bike Shop"

candidates = oracle.resolve_all_candidates(tags=tags, landmark_name=specific_name)

if len(candidates) == 1:
    print(f"✅ Success! State: Answerable")
    print(f"Target: {candidates[0]['name']} at {candidates[0]['coords']}")
else:
    print(f"State: Ambiguous/Contradictory. Found {len(candidates)} matches.")

🔍 Testing: Exact Named Entity Resolution
State: Ambiguous/Contradictory. Found 0 matches.


In [10]:
# 1. Broadest possible search (The "Where is Frank?" test)
query = "Franks Bike Shop".lower()
all_franks = oracle.poi_df[oracle.poi_df['name'].str.lower().str.contains(query, na=False)]

print(f"🔍 Absolute Name Search: Found {len(all_franks)} matches.")

# 2. Check the Tags for Frank
if not all_franks.empty:
    print("\nFrank's Metadata:")
    print(all_franks[['name', 'amenity', 'shop', 'brand']].iloc[0])
else:
    # Try a fuzzy match just in case of an apostrophe
    fuzzy_franks = oracle.poi_df[oracle.poi_df['name'].str.lower().str.contains("franks", na=False)]
    print(f"🔍 Fuzzy 'franks' Search: Found {len(fuzzy_franks)} matches.")

🔍 Absolute Name Search: Found 1 matches.

Frank's Metadata:
name       Franks Bike Shop
amenity                 NaN
shop                bicycle
brand                   NaN
Name: 2274, dtype: object


In [11]:
import pandas as pd
import config

def patched_resolve_all_candidates(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    """Temporary fix for the KeyError: 'semantic_score'"""
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        filtered_df = self.poi_df[
            (self.poi_df['y'] >= lat_min) & (self.poi_df['y'] <= lat_max) &
            (self.poi_df['x'] >= lon_min) & (self.poi_df['x'] <= lon_max)
        ].copy()
    else:
        filtered_df = self.poi_df.copy()

    if filtered_df.empty: return []

    # 1. Initialize the column immediately to prevent KeyError
    filtered_df['semantic_score'] = 1.0

    # 2. Category Filter (The "Inclusive" Fix)
    mask = pd.Series(False, index=filtered_df.index) # Start with all False
    
    # If no tags provided, let everyone through
    if not tags:
        mask = pd.Series(True, index=filtered_df.index)
    else:
        for key, value in tags.items():
            if key in filtered_df.columns:
                if value == "yes":
                    # Match if the column has ANY data
                    mask |= filtered_df[key].notna() & (filtered_df[key] != "")
                elif isinstance(value, list):
                    mask |= filtered_df[key].isin(value)
                else:
                    mask |= (filtered_df[key] == value)
    # 3. Semantic Scoring
    if landmark_name:
        query = landmark_name.lower().replace(" ", "_")
        name_hits = filtered_df['name'].astype(str).str.lower().str.contains(query.replace("_", " "), na=False)
        tag_hits = pd.Series(False, index=filtered_df.index)
        
        for col in config.POI_SEARCH_COLUMNS:
            if col in filtered_df.columns:
                tag_hits |= filtered_df[col].astype(str).str.lower().str.contains(query, na=False)
        
        filtered_df['semantic_score'] = (name_hits.astype(float) * 2.0) + (tag_hits.astype(float) * 1.5)

    # 4. Filter & Map
    candidates_df = filtered_df[filtered_df['semantic_score'] >= score_threshold]
    results = []
    for _, row in candidates_df.iterrows():
        raw_id = str(row['osmid']).replace('#', '')
        node_id = f"{self.prefix}{raw_id}"
        if node_id in self.G:
            results.append({
                "node_id": node_id, 
                "name": row.get('name', 'Unknown'), 
                "score": row['semantic_score'], 
                "coords": (row['y'], row['x'])
            })
    return results

# THE MAGIC LINE: Overwrite the method on the LIVE object
# Replace 'oracle' with whatever your OracleEngine variable name is
oracle.resolve_all_candidates = patched_resolve_all_candidates.__get__(oracle, oracle.__class__)

print("✅ Oracle method patched in memory. Try running your test again!")

✅ Oracle method patched in memory. Try running your test again!


In [12]:
# Testing Specific Identity Resolution
print("🔍 Testing: Named Entity Resolution")
tags = config.LANDMARK_GROUPS["BIKE"]
name = "East Village bicycle shop"
candidates = oracle.resolve_all_candidates(tags=tags, landmark_name=name)

# LOGIC: Because the name is highly specific, candidates should be 1.
state = "Answerable" if len(candidates) == 1 else "Ambiguous"
print(f"State: {state} | Candidates: {len(candidates)}")

🔍 Testing: Named Entity Resolution
State: Ambiguous | Candidates: 0


In [16]:
def final_patched_resolve(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    # 1. Spatial Pruning (Stayed the same)
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        filtered_df = self.poi_df[
            (self.poi_df['y'] >= lat_min) & (self.poi_df['y'] <= lat_max) &
            (self.poi_df['x'] >= lon_min) & (self.poi_df['x'] <= lon_max)
        ].copy()
    else:
        filtered_df = self.poi_df.copy()

    if filtered_df.empty: return []

    # 2. CATEGORY FILTER (The "Null-Safe Inclusive" Fix)
    # Start with all False, then 'OR' in the matches
    mask = pd.Series(False, index=filtered_df.index)
    
    if not tags:
        mask = pd.Series(True, index=filtered_df.index)
    else:
        for key, val in tags.items():
            if key in filtered_df.columns:
                # Use .fillna() to ensure we don't lose rows to NaN comparisons
                col_data = filtered_df[key].fillna("")
                
                if val == "yes":
                    mask |= (col_data != "")
                elif isinstance(val, list):
                    mask |= col_data.isin(val)
                else:
                    mask |= (col_data == val)
    
    filtered_df = filtered_df[mask].copy()
    if filtered_df.empty: return []

    # 3. SEMANTIC SCORING (The "Frank-Finder")
    filtered_df['semantic_score'] = 1.0
    if landmark_name and landmark_name != "[MASK]":
        query = landmark_name.lower()
        # Clean the name column for comparison
        names_clean = filtered_df['name'].fillna("").str.lower()
        name_hits = names_clean.str.contains(query, na=False)
        
        tag_hits = pd.Series(False, index=filtered_df.index)
        import config
        for col in config.POI_SEARCH_COLUMNS:
            if col in filtered_df.columns:
                tag_hits |= filtered_df[col].fillna("").astype(str).str.lower().str.contains(query, na=False)
        
        filtered_df['semantic_score'] = (name_hits.astype(float) * 2.0) + (tag_hits.astype(float) * 1.5)

    # 4. FINAL MAPPING
    candidates_df = filtered_df[filtered_df['semantic_score'] >= score_threshold]
    results = []
    for _, row in candidates_df.iterrows():
        raw_id = str(row['osmid']).replace('#', '')
        node_id = f"{self.prefix}{raw_id}"
        if node_id in self.G:
            results.append({
                "node_id": node_id,
                "name": row.get('name', 'Unknown'),
                "score": row['semantic_score'],
                "coords": (row['y'], row['x'])
            })
    return results

# Re-Apply the patch to your live object
oracle.resolve_all_candidates = final_patched_resolve.__get__(oracle, oracle.__class__)
print("✅ Final Null-Safe Patch Applied.")

✅ Final Null-Safe Patch Applied.


In [17]:
# Testing Specific Identity Resolution
print("🔍 Testing: Named Entity Resolution")
tags = config.LANDMARK_GROUPS["BIKE"]
name = "East Village bicycle shop"
candidates = oracle.resolve_all_candidates(tags=tags, landmark_name=name)

# LOGIC: Because the name is highly specific, candidates should be 1.
state = "Answerable" if len(candidates) == 1 else "Ambiguous"
print(f"State: {state} | Candidates: {len(candidates)}")

🔍 Testing: Named Entity Resolution
State: Ambiguous | Candidates: 0


In [ ]:
# Testing with the ACTUAL name found in the POI list
print("🔍 Testing: Actual Entity Match")
tags = config.LANDMARK_GROUPS["BIKE"]
# We use 'Franks' because we SAW it in our 'Absolute Name Search'
true_name = "Franks Bike Shop" 

candidates = oracle.resolve_all_candidates(tags=tags, landmark_name=true_name)

if len(candidates) == 1:
    print(f"✅ Success! State: Answerable")
    print(f"Node: {candidates[0]['node_id']} | Name: {candidates[0]['name']}")
else:
    print(f"❌ Still 0? Check if node_id {candidates[0]['node_id'] if candidates else 'N/A'} is in self.G")

🔍 Testing: Actual Entity Match
✅ Success! State: Answerable
Node: 1#483978210 | Name: Franks Bike Shop


In [19]:
def final_patched_resolve(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    # 1. Spatial Pruning (Stayed the same)
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        filtered_df = self.poi_df[
            (self.poi_df['y'] >= lat_min) & (self.poi_df['y'] <= lat_max) &
            (self.poi_df['x'] >= lon_min) & (self.poi_df['x'] <= lon_max)
        ].copy()
    else:
        filtered_df = self.poi_df.copy()

    if filtered_df.empty: return []

    # 2. CATEGORY FILTER (The "Null-Safe Inclusive" Fix)
    # Start with all False, then 'OR' in the matches
    mask = pd.Series(False, index=filtered_df.index)
    
    if not tags:
        mask = pd.Series(True, index=filtered_df.index)
    else:
        for key, val in tags.items():
            if key in filtered_df.columns:
                # Use .fillna() to ensure we don't lose rows to NaN comparisons
                col_data = filtered_df[key].fillna("")
                
                if val == "yes":
                    mask |= (col_data != "")
                elif isinstance(val, list):
                    mask |= col_data.isin(val)
                else:
                    mask |= (col_data == val)
    
    filtered_df = filtered_df[mask].copy()
    if filtered_df.empty: return []

    # 3. SEMANTIC SCORING (The "Frank-Finder")
    # --- STEP 3: IMPROVED SEMANTIC SCORING ---
    if landmark_name and landmark_name != "[MASK]":
        query = landmark_name.lower()
        
        # 1. Check the Name Column (Weight: 2.0)
        name_hits = filtered_df['name'].fillna("").str.lower().str.contains(query, na=False)
        
        # 2. Check the Suburb/Neighborhood (Weight: 1.5)
        suburb_hits = pd.Series(False, index=filtered_df.index)
        if 'addr:suburb' in filtered_df.columns:
            suburb_hits = filtered_df['addr:suburb'].fillna("").str.lower().str.contains(query, na=False)
        
        # 3. Check the Street (Weight: 1.5)
        street_hits = pd.Series(False, index=filtered_df.index)
        if 'addr:street' in filtered_df.columns:
            street_hits = filtered_df['addr:street'].fillna("").str.lower().str.contains(query, na=False)

        # 4. Combine them!
        # If ANY of these hit, the landmark is a candidate.
        filtered_df['semantic_score'] = (
            (name_hits.astype(float) * 2.0) + 
            (suburb_hits.astype(float) * 1.5) + 
            (street_hits.astype(float) * 1.5)
        )

    # 4. FINAL MAPPING
    candidates_df = filtered_df[filtered_df['semantic_score'] >= score_threshold]
    results = []
    for _, row in candidates_df.iterrows():
        raw_id = str(row['osmid']).replace('#', '')
        node_id = f"{self.prefix}{raw_id}"
        if node_id in self.G:
            results.append({
                "node_id": node_id,
                "name": row.get('name', 'Unknown'),
                "score": row['semantic_score'],
                "coords": (row['y'], row['x'])
            })
    return results

# Re-Apply the patch to your live object
oracle.resolve_all_candidates = final_patched_resolve.__get__(oracle, oracle.__class__)
print("✅ Final Null-Safe Patch Applied.")

✅ Final Null-Safe Patch Applied.


In [20]:
print("🔍 Testing: Strategic Split Search")
# We look for the CATEGORY in the tags, and the LOCATION in the name field
tags = config.LANDMARK_GROUPS["BIKE"]
name_modifier = "East Village" 

candidates = oracle.resolve_all_candidates(tags=tags, landmark_name=name_modifier)

print(f"Found {len(candidates)} bike places in the East Village.")
for c in candidates:
    print(f" - {c['name']} (Score: {c['score']})")

🔍 Testing: Strategic Split Search
Found 1 bike places in the East Village.
 - East Village (Score: 2.0)


In [23]:
def final_patched_resolve(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    # 1. SPATIAL PRUNING
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        filtered_df = self.poi_df[
            (self.poi_df['y'] >= lat_min) & (self.poi_df['y'] <= lat_max) &
            (self.poi_df['x'] >= lon_min) & (self.poi_df['x'] <= lon_max)
        ].copy()
    else:
        filtered_df = self.poi_df.copy()

    if filtered_df.empty: return []

    # 2. CATEGORY FILTER (STRICT)
    category_mask = pd.Series(False, index=filtered_df.index)
    if not tags:
        category_mask = pd.Series(True, index=filtered_df.index)
    else:
        for key, val in tags.items():
            if key in filtered_df.columns:
                col_data = filtered_df[key].fillna("")
                if val == "yes":
                    category_mask |= (col_data != "")
                elif isinstance(val, list):
                    category_mask |= col_data.isin(val)
                else:
                    mask_val = (col_data == val)
                    category_mask |= mask_val
    
    filtered_df = filtered_df[category_mask].copy()
    if filtered_df.empty: return []

    # 3. SEMANTIC SCORING
    # Default score for [MASK] or category-only searches
    filtered_df['semantic_score'] = 1.0

    if landmark_name and landmark_name != "[MASK]":
        query = landmark_name.lower()
        
        # Check specific columns
        name_hits = filtered_df['name'].fillna("").str.lower().str.contains(query, na=False)
        suburb_hits = filtered_df.get('addr:suburb', pd.Series("", index=filtered_df.index)).fillna("").str.lower().str.contains(query, na=False)
        street_hits = filtered_df.get('addr:street', pd.Series("", index=filtered_df.index)).fillna("").str.lower().str.contains(query, na=False)

        # Final score calculation
        filtered_df['semantic_score'] = (name_hits.astype(float) * 2.0) + \
                                        (suburb_hits.astype(float) * 1.5) + \
                                        (street_hits.astype(float) * 1.5)
        
        # CRITICAL: If they provided a name, but we found no string match, 
        # then this node is NOT a candidate for this specific search.
        any_match = name_hits | suburb_hits | street_hits
        filtered_df = filtered_df[any_match].copy()

    if filtered_df.empty: return []

    # 4. FINAL MAPPING
    candidates_df = filtered_df[filtered_df['semantic_score'] >= score_threshold]
    results = []
    for _, row in candidates_df.iterrows():
        raw_id = str(row['osmid']).replace('#', '')
        node_id = f"{self.prefix}{raw_id}"
        if node_id in self.G:
            results.append({
                "node_id": node_id,
                "name": row.get('name', 'Unknown'),
                "score": row['semantic_score'],
                "coords": (row['y'], row['x'])
            })
    return results

# Re-Apply the patch
oracle.resolve_all_candidates = final_patched_resolve.__get__(oracle, oracle.__class__)
print("✅ Consolidated Patch Applied. Duplicates removed.")

✅ Consolidated Patch Applied. Duplicates removed.


In [24]:
print("🔍 Testing: Strategic Split Search")
# We look for the CATEGORY in the tags, and the LOCATION in the name field
tags = config.LANDMARK_GROUPS["BIKE"]
name_modifier = "East Village" 

candidates = oracle.resolve_all_candidates(tags=tags, landmark_name=name_modifier)

print(f"Found {len(candidates)} bike places in the East Village.")
for c in candidates:
    print(f" - {c['name']} (Score: {c['score']})")

🔍 Testing: Strategic Split Search
Found 1 bike places in the East Village.
 - East Village (Score: 2.0)


In [ ]:
# Find that specific "East Village" node that keeps appearing
ev_node = oracle.poi_df[oracle.poi_df['name'] == "East Village"]

print("📊 Metadata for the 'East Village' node:")
# Display only the columns that have actual data
print(ev_node.dropna(axis=1).iloc[0])

# Check if any of these keys match our BIKE config
import config
bike_tags = config.LANDMARK_GROUPS["BIKE"]
print(f"\nTarget BIKE Tags: {bike_tags}")

📊 Metadata for the 'East Village' node:
unique_id                       node/158842996
osmid                               #158842996
element_type                              node
name                              East Village
geometry        POINT (-73.9873613 40.7292688)
centroid        POINT (-73.9873613 40.7292688)
cellids                  [9926595056351838208]
x                                   -73.987361
y                                    40.729269
clean_name                         eastvillage
Name: 13, dtype: object

Target BIKE Tags: {'amenity': ['bicycle_parking', 'bicycle_rental'], 'shop': ['bicycle', 'yes'], 'brand': 'yes'}


In [26]:
def final_patched_resolve(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    # 1. SPATIAL PRUNING
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        filtered_df = self.poi_df[
            (self.poi_df['y'] >= lat_min) & (self.poi_df['y'] <= lat_max) &
            (self.poi_df['x'] >= lon_min) & (self.poi_df['x'] <= lon_max)
        ].copy()
    else:
        filtered_df = self.poi_df.copy()

    if filtered_df.empty: return []

    # --- 2. CATEGORY FILTER (FIXED) ---
    if not tags or len(tags) == 0:
        # If no tags, everything is potentially valid (use with caution!)
        category_mask = pd.Series(True, index=filtered_df.index)
    else:
        category_mask = pd.Series(False, index=filtered_df.index)
        for key, val in tags.items():
            if key in filtered_df.columns:
                # Fill NaN with a value that will NEVER match our search
                col_data = filtered_df[key].fillna("N/A")
                
                if val == "yes":
                    category_mask |= (col_data != "N/A")
                elif isinstance(val, list):
                    category_mask |= col_data.isin(val)
                else:
                    category_mask |= (col_data == val)
    
    # APPLY THE GATE: If it's not a bike/cafe/etc, it dies here.
    filtered_df = filtered_df[category_mask].copy()
    
    # 3. SEMANTIC SCORING
    # Default score for [MASK] or category-only searches
    filtered_df['semantic_score'] = 1.0

    if landmark_name and landmark_name != "[MASK]":
        query = landmark_name.lower()
        
        # Check specific columns
        name_hits = filtered_df['name'].fillna("").str.lower().str.contains(query, na=False)
        suburb_hits = filtered_df.get('addr:suburb', pd.Series("", index=filtered_df.index)).fillna("").str.lower().str.contains(query, na=False)
        street_hits = filtered_df.get('addr:street', pd.Series("", index=filtered_df.index)).fillna("").str.lower().str.contains(query, na=False)

        # Final score calculation
        filtered_df['semantic_score'] = (name_hits.astype(float) * 2.0) + \
                                        (suburb_hits.astype(float) * 1.5) + \
                                        (street_hits.astype(float) * 1.5)
        
        # CRITICAL: If they provided a name, but we found no string match, 
        # then this node is NOT a candidate for this specific search.
        any_match = name_hits | suburb_hits | street_hits
        filtered_df = filtered_df[any_match].copy()

    if filtered_df.empty: return []

    # 4. FINAL MAPPING
    candidates_df = filtered_df[filtered_df['semantic_score'] >= score_threshold]
    results = []
    for _, row in candidates_df.iterrows():
        raw_id = str(row['osmid']).replace('#', '')
        node_id = f"{self.prefix}{raw_id}"
        if node_id in self.G:
            results.append({
                "node_id": node_id,
                "name": row.get('name', 'Unknown'),
                "score": row['semantic_score'],
                "coords": (row['y'], row['x'])
            })
    return results

# Re-Apply the patch
oracle.resolve_all_candidates = final_patched_resolve.__get__(oracle, oracle.__class__)
print("✅ Consolidated Patch Applied. Duplicates removed.")

✅ Consolidated Patch Applied. Duplicates removed.


In [27]:
print("🔍 Testing: The neighborhood should now be BLOCKED")
tags = config.LANDMARK_GROUPS["BIKE"]
name_mod = "East Village"

results = oracle.resolve_all_candidates(tags=tags, landmark_name=name_mod)

if any(r['name'] == "East Village" for r in results):
    print("❌ ERROR: The neighborhood node survived the category filter!")
elif len(results) == 0:
    print("✅ SUCCESS: The neighborhood was blocked. Now we just need to find Frank.")
else:
    print(f"✅ SUCCESS: Found {len(results)} valid bike entities.")

🔍 Testing: The neighborhood should now be BLOCKED
❌ ERROR: The neighborhood node survived the category filter!


In [28]:
import pandas as pd
import config

# 1. Get the neighborhood node
ev_node = oracle.poi_df[oracle.poi_df['name'] == "East Village"].iloc[0:1]
tags = config.LANDMARK_GROUPS["BIKE"]

print(f"--- Diagnostic for Node: {ev_node['name'].values[0]} ---")
for key, val in tags.items():
    if key in ev_node.columns:
        col_val = ev_node[key].values[0]
        # Check the logic manually
        match = False
        if val == "yes":
            match = pd.notna(col_val) and col_val != ""
        elif isinstance(val, list):
            match = col_val in val
        else:
            match = col_val == val
            
        print(f"Key: [{key}] | Node Value: [{col_val}] | Search Value: [{val}] | MATCH: {match}")

--- Diagnostic for Node: East Village ---
Key: [amenity] | Node Value: [nan] | Search Value: [['bicycle_parking', 'bicycle_rental']] | MATCH: False
Key: [shop] | Node Value: [nan] | Search Value: [['bicycle', 'yes']] | MATCH: False
Key: [brand] | Node Value: [nan] | Search Value: [yes] | MATCH: False


In [29]:
def final_patched_resolve(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    # 1. SPATIAL PRUNING
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        filtered_df = self.poi_df[
            (self.poi_df['y'] >= lat_min) & (self.poi_df['y'] <= lat_max) &
            (self.poi_df['x'] >= lon_min) & (self.poi_df['x'] <= lon_max)
        ].copy()
    else:
        filtered_df = self.poi_df.copy()

    if filtered_df.empty: return []

    # 2. CATEGORY FILTER (STRICT TYPE-SAFE)
    if tags:
        # Start with all False
        category_mask = pd.Series(False, index=filtered_df.index)
        
        for key, val in tags.items():
            if key in filtered_df.columns:
                s = filtered_df[key].fillna("N/A")
                if val == "yes":
                    category_mask |= (s != "N/A") & (s != "")
                elif isinstance(val, list):
                    # USE .isin() EXPLICITLY
                    category_mask |= s.isin(val)
                else:
                    category_mask |= (s == val)
        
        filtered_df = filtered_df[category_mask].copy()

    # 3. AREA PROTECTION (No Neighborhoods allowed to be Shops)
    if 'place' in filtered_df.columns:
        filtered_df = filtered_df[~filtered_df['place'].isin(['suburb', 'neighbourhood', 'village'])].copy()

    if filtered_df.empty: return []

    # 4. SEMANTIC SCORING
    filtered_df['semantic_score'] = 1.0 # Default for [MASK]
    
    if landmark_name and landmark_name != "[MASK]":
        query = landmark_name.lower()
        
        # Calculate hits
        name_hits = filtered_df['name'].fillna("").str.lower().str.contains(query, na=False)
        sub_hits = filtered_df.get('addr:suburb', pd.Series("", index=filtered_df.index)).fillna("").str.lower().str.contains(query, na=False)
        str_hits = filtered_df.get('addr:street', pd.Series("", index=filtered_df.index)).fillna("").str.lower().str.contains(query, na=False)

        # Apply Weights
        filtered_df['semantic_score'] = (name_hits.astype(float) * 2.0) + (sub_hits.astype(float) * 1.5) + (str_hits.astype(float) * 1.5)
        
        # DROP ANYONE WHO DOESN'T HAVE A NAME MATCH
        filtered_df = filtered_df[(name_hits | sub_hits | str_hits)].copy()

    if filtered_df.empty: return []

    # 5. MAPPING
    candidates_df = filtered_df[filtered_df['semantic_score'] >= score_threshold]
    results = []
    for _, row in candidates_df.iterrows():
        node_id = f"{self.prefix}{str(row['osmid']).replace('#', '')}"
        if node_id in self.G:
            results.append({
                "node_id": node_id, "name": row.get('name', 'Unknown'),
                "score": row['semantic_score'], "coords": (row['y'], row['x'])
            })
    return sorted(results, key=lambda x: x['score'], reverse=True)

# Re-patch
oracle.resolve_all_candidates = final_patched_resolve.__get__(oracle, oracle.__class__)
print("✅ Fortress Patch Applied. Neighborhoods are now logically impossible.")

✅ Fortress Patch Applied. Neighborhoods are now logically impossible.


In [31]:
test_cases = [
    {"name": "Unique Entity", "tags": config.LANDMARK_GROUPS["BIKE"], "query": "Franks Bike Shop", "expected": "Answerable"},
    {"name": "Neighborhood Mask", "tags": config.LANDMARK_GROUPS["BIKE"], "query": "East Village", "expected": "Ambiguous/Multiple"},
    {"name": "Pure [MASK]", "tags": config.LANDMARK_GROUPS["BIKE"], "query": "[MASK]", "expected": "Ambiguous/Broad"},
    {"name": "Contradiction", "tags": config.LANDMARK_GROUPS["CAFE"], "query": "Space Shuttle", "expected": "Empty"}
]

for tc in test_cases:
    res = oracle.resolve_all_candidates(tags=tc['tags'], landmark_name=tc['query'])
    print(f"Test: {tc['name']} | Found: {len(res)} | Result: {res[0]['name'] if res else 'None'}")

Test: Unique Entity | Found: 1 | Result: Franks Bike Shop
Test: Neighborhood Mask | Found: 1 | Result: East Village
Test: Pure [MASK] | Found: 4952 | Result: Five Guys
Test: Contradiction | Found: 0 | Result: None


In [32]:
def titanium_fortress_resolve(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    """
    Final optimized resolver for RVS dataset generation.
    Handles:
    - Spatial Pruning
    - Neighborhood/Area Exclusion
    - Strict Category Filtering (OR within tags)
    - Multi-column Semantic Scoring (Name + Address)
    """
    # 1. SPATIAL PRUNING
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        df = self.poi_df[
            (self.poi_df['y'] >= lat_min) & (self.poi_df['y'] <= lat_max) &
            (self.poi_df['x'] >= lon_min) & (self.poi_df['x'] <= lon_max)
        ].copy()
    else:
        df = self.poi_df.copy()

    if df.empty: return []

    # 2. THE "NOT-A-POI" FILTER (The Neighborhood Killer)
    # We explicitly remove nodes that represent regions rather than specific locations
    if 'place' in df.columns:
        invalid_places = ['suburb', 'neighbourhood', 'village', 'city_district', 'borough', 'city']
        df = df[~df['place'].fillna("").isin(invalid_places)].copy()
    
    if df.empty: return []

    # 3. CATEGORY GATE
    if tags:
        category_mask = pd.Series(False, index=df.index)
        for key, val in tags.items():
            if key in df.columns:
                s = df[key].fillna("N/A")
                if val == "yes":
                    category_mask |= (s != "N/A") & (s != "")
                elif isinstance(val, list):
                    category_mask |= s.isin(val)
                else:
                    category_mask |= (s == val)
        
        # Apply the Gate: Must be a Bike Shop/Cafe/etc.
        df = df[category_mask].copy()

    if df.empty: return []

    # 4. SEMANTIC SCORING
    df['semantic_score'] = 1.0 # Default score
    
    if landmark_name and landmark_name != "[MASK]":
        query = landmark_name.lower()
        
        # Calculate column-specific hits
        name_hits = df['name'].fillna("").str.lower().str.contains(query, na=False)
        sub_hits = df.get('addr:suburb', pd.Series("", index=df.index)).fillna("").str.lower().str.contains(query, na=False)
        str_hits = df.get('addr:street', pd.Series("", index=df.index)).fillna("").str.lower().str.contains(query, na=False)
        neighborhood_hits = df.get('addr:neighborhood', pd.Series("", index=df.index)).fillna("").str.lower().str.contains(query, na=False)

        # Apply Weights (Identity > Location)
        df['semantic_score'] = (name_hits.astype(float) * 2.5) + \
                               (sub_hits.astype(float) * 1.5) + \
                               (str_hits.astype(float) * 1.5) + \
                               (neighborhood_hits.astype(float) * 1.5)
        
        # DROPPING NON-MATCHES: If a name was provided, the entity MUST match it somehow
        any_name_match = name_hits | sub_hits | str_hits | neighborhood_hits
        df = df[any_name_match].copy()

    if df.empty: return []

    # 5. MAPPING TO NETWORKX NODES
    candidates_df = df[df['semantic_score'] >= score_threshold]
    results = []
    for _, row in candidates_df.iterrows():
        # Handle the OSMID prefixing
        raw_id = str(row['osmid']).replace('#', '')
        node_id = f"{self.prefix}{raw_id}"
        
        if node_id in self.G:
            results.append({
                "node_id": node_id,
                "name": row.get('name', 'Unknown'),
                "score": row['semantic_score'],
                "coords": (row['y'], row['x'])
            })
            
    # Sort by score descending so the best match is index 0
    return sorted(results, key=lambda x: x['score'], reverse=True)

# APPLY THE PATCH
import types
oracle.resolve_all_candidates = types.MethodType(titanium_fortress_resolve, oracle)
print("🛡️ Titanium Fortress Patch Applied! Run your tests now.")

🛡️ Titanium Fortress Patch Applied! Run your tests now.


In [33]:
test_cases = [
    {"name": "Unique Entity", "tags": config.LANDMARK_GROUPS["BIKE"], "query": "Franks Bike Shop", "expected": "Answerable"},
    {"name": "Neighborhood Mask", "tags": config.LANDMARK_GROUPS["BIKE"], "query": "East Village", "expected": "Ambiguous/Multiple"},
    {"name": "Pure [MASK]", "tags": config.LANDMARK_GROUPS["BIKE"], "query": "[MASK]", "expected": "Ambiguous/Broad"},
    {"name": "Contradiction", "tags": config.LANDMARK_GROUPS["CAFE"], "query": "Space Shuttle", "expected": "Empty"}
]

for tc in test_cases:
    res = oracle.resolve_all_candidates(tags=tc['tags'], landmark_name=tc['query'])
    print(f"Test: {tc['name']} | Found: {len(res)} | Result: {res[0]['name'] if res else 'None'}")

Test: Unique Entity | Found: 1 | Result: Franks Bike Shop
Test: Neighborhood Mask | Found: 1 | Result: East Village
Test: Pure [MASK] | Found: 4952 | Result: Five Guys
Test: Contradiction | Found: 0 | Result: None


In [34]:
# Target the specific node that keeps surviving
immortal_node = oracle.poi_df[oracle.poi_df['name'] == "East Village"]

print("🔍 IMMORTAL NODE CHECK")
print(f"Place tag value: '{immortal_node['place'].values[0]}'")
print(f"All non-null tags for this node:")
print(immortal_node.dropna(axis=1).iloc[0])

🔍 IMMORTAL NODE CHECK
Place tag value: 'neighbourhood'
All non-null tags for this node:
unique_id                       node/158842996
osmid                               #158842996
element_type                              node
name                              East Village
geometry        POINT (-73.9873613 40.7292688)
centroid        POINT (-73.9873613 40.7292688)
cellids                  [9926595056351838208]
x                                   -73.987361
y                                    40.729269
clean_name                         eastvillage
Name: 13, dtype: object


There it is! The "British Spelling" trap: neighboUrhood

In [35]:
def final_rvs_resolver(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    # 1. SPATIAL PRUNING
    df = self.poi_df.copy()
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        df = df[(df['y'] >= lat_min) & (df['y'] <= lat_max) &
                (df['x'] >= lon_min) & (df['x'] <= lon_max)].copy()
    if df.empty: return []

    # 2. THE AREA KILLER (Now with both spellings!)
    if 'place' in df.columns:
        # We kill anything that is a region/area
        invalid = ['suburb', 'neighbourhood', 'neighborhood', 'village', 'city_district', 'borough']
        df = df[~df['place'].fillna("").isin(invalid)].copy()

    # 3. THE CATEGORY GATE (Mandatory for RVS)
    # If we are looking for a BIKE shop, the node MUST have a bike-related tag.
    if tags:
        category_mask = pd.Series(False, index=df.index)
        for key, val in tags.items():
            if key in df.columns:
                s = df[key].fillna("N/A")
                if val == "yes":
                    category_mask |= (s != "N/A") & (s != "")
                elif isinstance(val, list):
                    category_mask |= s.isin(val)
                else:
                    category_mask |= (s == val)
        
        # Applying the filter - This WILL kill the East Village node 
        # because it has NaN for shop and amenity.
        df = df[category_mask].copy()

    if df.empty: return []

    # 4. SEMANTIC SCORING
    df['semantic_score'] = 1.0
    if landmark_name and landmark_name != "[MASK]":
        query = landmark_name.lower()
        name_hits = df['name'].fillna("").str.lower().str.contains(query, na=False)
        sub_hits = df.get('addr:suburb', pd.Series("", index=df.index)).fillna("").str.lower().str.contains(query, na=False)
        
        # Scoring weights
        df['semantic_score'] = (name_hits.astype(float) * 2.5) + (sub_hits.astype(float) * 1.5)
        
        # Only keep rows that matched the name query somehow
        df = df[(name_hits | sub_hits)].copy()

    # 5. MAPPING
    results = []
    for _, row in df[df['semantic_score'] >= score_threshold].iterrows():
        node_id = f"{self.prefix}{str(row['osmid']).replace('#', '')}"
        if node_id in self.G:
            results.append({
                "node_id": node_id, "name": row.get('name', 'Unknown'),
                "score": row['semantic_score'], "coords": (row['y'], row['x'])
            })
    return sorted(results, key=lambda x: x['score'], reverse=True)

# Patch it one last time
oracle.resolve_all_candidates = final_rvs_resolver.__get__(oracle, oracle.__class__)
print("🎯 Final RVS Resolver Active. Neighbourhood (with a 'u') is now blocked.")

🎯 Final RVS Resolver Active. Neighbourhood (with a 'u') is now blocked.


In [37]:
# Final Verification for RVS Logic
import pandas as pd

test_cases = [
    {"name": "Unique Entity", "tags": config.LANDMARK_GROUPS["BIKE"], "query": "Franks Bike Shop"},
    {"name": "Neighborhood Mask", "tags": config.LANDMARK_GROUPS["BIKE"], "query": "East Village"},
    {"name": "Pure [MASK]", "tags": config.LANDMARK_GROUPS["BIKE"], "query": "[MASK]"},
    {"name": "Contradiction", "tags": config.LANDMARK_GROUPS["CAFE"], "query": "Space Shuttle"}
]

print(f"{'TEST NAME':<20} | {'FOUND':<5} | {'TOP RESULT':<25} | {'SCORE':<5}")
print("-" * 65)

for tc in test_cases:
    # Execute the resolver
    candidates = oracle.resolve_all_candidates(tags=tc['tags'], landmark_name=tc['query'])
    
    found_count = len(candidates)
    top_name = candidates[0]['name'] if found_count > 0 else "None"
    top_score = candidates[0]['score'] if found_count > 0 else 0.0
    
    print(f"{tc['name']:<20} | {found_count:<5} | {top_name:<25} | {top_score:<5.1f}")

# --- CRITICAL SMOKE TEST ---
# Let's specifically check if the neighborhood node (ID: 158842996) is in the results
ev_mask_results = oracle.resolve_all_candidates(tags=config.LANDMARK_GROUPS["BIKE"], landmark_name="East Village")
is_neighborhood_present = any("158842996" in c['node_id'] for c in ev_mask_results)

if is_neighborhood_present:
    print("\n❌ FAILED: The neighborhood node STILL smuggled itself through.")
else:
    print("\n✅ PASSED: The neighborhood node is officially blocked.")

TEST NAME            | FOUND | TOP RESULT                | SCORE
-----------------------------------------------------------------
Unique Entity        | 1     | Franks Bike Shop          | 2.5  
Neighborhood Mask    | 1     | East Village              | 2.5  
Pure [MASK]          | 4952  | Five Guys                 | 1.0  
Contradiction        | 0     | None                      | 0.0  

✅ PASSED: The neighborhood node is officially blocked.


In [38]:
# Who is this other East Village that is a bike entity?
secret_node = oracle.poi_df[(oracle.poi_df['name'] == "East Village") & (oracle.poi_df['osmid'] != 158842996)]
print(secret_node.dropna(axis=1))

             unique_id        osmid element_type          name  \
13      node/158842996   #158842996         node  East Village   
12855  node/6621617760  #6621617760         node  East Village   

                         geometry                        centroid  \
13     POINT (-73.98736 40.72927)  POINT (-73.9873613 40.7292688)   
12855  POINT (-73.99081 40.72945)  POINT (-73.9908134 40.7294535)   

                     cellids          x          y   clean_name  
13     [9926595056351838208] -73.987361  40.729269  eastvillage  
12855  [9926595053701038080] -73.990813  40.729453  eastvillage  


After updating oracle:

In [39]:
# 1. Initialize the Oracle with your data
# oracle = OracleEngine(G, poi_df) 

print("🚀 Starting Oracle Patch Verification...\n")

# --- TEST 1: The "Neighbourhood" Killer (Regex Check) ---
# We use the specific ID of the East Village neighborhood node
ev_id = "node/158842996" 
tags = config.LANDMARK_GROUPS["BIKE"]
res_ev = oracle.resolve_all_candidates(tags=tags, landmark_name="East Village")

is_blocked = not any(ev_id in r['node_id'] for r in res_ev)
found_count = len(res_ev)

print(f"Test 1 [Area Exclusion]: {'✅ PASSED' if is_blocked else '❌ FAILED'}")
print(f"   -> Neighborhood node {ev_id} was successfully blocked.")
print(f"   -> Still found {found_count} valid POI(s) matching 'East Village'.")

# --- TEST 2: Semantic Scoring (Address vs Name) ---
# This checks if the 2.5 (Name) + 1.5 (Address) weighting works
franks_res = oracle.resolve_all_candidates(tags=tags, landmark_name="Franks Bike Shop")

if franks_res and franks_res[0]['name'] == "Franks Bike Shop":
    print(f"Test 2 [Semantic Weight]: ✅ PASSED (Score: {franks_res[0]['score']})")
else:
    print(f"Test 2 [Semantic Weight]: ❌ FAILED (Top result: {franks_res[0]['name'] if franks_res else 'None'})")

# --- TEST 3: The [MASK] Broad Search ---
# Ensures the code still works when no name is provided (for underspecification)
mask_res = oracle.resolve_all_candidates(tags=tags, landmark_name="[MASK]")
print(f"Test 3 [Global Recall]: {'✅ PASSED' if len(mask_res) > 10 else '❌ FAILED'}")
print(f"   -> Found {len(mask_res)} total bike entities in the city.")

# --- TEST 4: Spatial Pruning ---
# Tests if the bounds parameter in the new code actually restricts results
# (Using a tiny box around Frank's coordinates)
if franks_res:
    fy, fx = franks_res[0]['coords']
    small_bounds = (fy - 0.001, fy + 0.001, fx - 0.001, fx + 0.001)
    local_res = oracle.resolve_all_candidates(tags=tags, bounds=small_bounds)
    print(f"Test 4 [Spatial Pruning]: {'✅ PASSED' if len(local_res) < len(mask_res) else '❌ FAILED'}")

🚀 Starting Oracle Patch Verification...

Test 1 [Area Exclusion]: ✅ PASSED
   -> Neighborhood node node/158842996 was successfully blocked.
   -> Still found 1 valid POI(s) matching 'East Village'.
Test 2 [Semantic Weight]: ✅ PASSED (Score: 2.5)
Test 3 [Global Recall]: ✅ PASSED
   -> Found 4952 total bike entities in the city.
Test 4 [Spatial Pruning]: ✅ PASSED


In [ ]:
# Pittsburgh Configuration Mapping
PITTS_CONFIG = {
    "BOUTIQUE": {"shop": ["clothes", "boutique", "gift"]},
    "SCHOOL": {"amenity": ["school", "college", "university"]},
    "FITNESS": {"leisure": "fitness_centre", "amenity": "gym"},
    "GARDEN": {"leisure": ["garden", "park", "pitch"]},
}

pittsburgh_tests = [
    {
        "name": "Specific Name vs Type", 
        "tags": PITTS_CONFIG["BOUTIQUE"], 
        "query": "Love, Pittsburgh", 
        "note": "Sample 8112: Checks if comma/punctuation in names breaks the regex."
    },
    {
        "name": "Semantic Overlap (Pitch)", 
        "tags": PITTS_CONFIG["GARDEN"], 
        "query": "Bing pitch", 
        "note": "Sample 8111: Checks if 'pitch' (sport) is caught in a 'garden' category search."
    },
    {
        "name": "British vs US Spelling", 
        "tags": PITTS_CONFIG["FITNESS"], 
        "query": "fitness center", 
        "note": "Sample 8109: The query uses 'center' but OSM often uses 'centre'. Checks the fuzzy/loc_hits logic."
    },
    {
        "name": "Neighborhood Address Hit", 
        "tags": PITTS_CONFIG["BOUTIQUE"], 
        "query": "East Carson Street", 
        "note": "Sample 8107: Tests if loc_hits correctly finds shops by street name alone."
    }
]

print(f"{'TEST NAME':<25} | {'FOUND':<5} | {'TOP RESULT':<25} | {'SCORE':<5}")
print("-" * 75)

for tc in pittsburgh_tests:
    res = oracle.resolve_all_candidates(tags=tc['tags'], landmark_name=tc['query'])
    found = len(res)
    top_name = res[0]['name'] if found > 0 else "None"
    top_score = res[0]['score'] if found > 0 else 0.0
    
    print(f"{tc['name']:<25} | {found:<5} | {top_name:<25} | {top_score:<5.1f}")

TEST NAME                 | FOUND | TOP RESULT                | SCORE
---------------------------------------------------------------------------
Specific Name vs Type     | 0     | None                      | 0.0  
Semantic Overlap (Pitch)  | 0     | None                      | 0.0  
British vs US Spelling    | 0     | None                      | 0.0  
Neighborhood Address Hit  | 0     | None                      | 0.0  


In [41]:
# Check the "Love, Pittsburgh" node specifically
target = oracle.poi_df[oracle.poi_df['name'].str.contains("Love, Pittsburgh", na=False)]

print("🔍 DEBUGGING PITTSBURGH POI")
if target.empty:
    print("❌ Node not found by name at all! Check if the POI file is loaded.")
else:
    print(f"Node Found! Columns available: {target.dropna(axis=1).columns.tolist()}")
    print(f"Shop tag: {target['shop'].values[0] if 'shop' in target.columns else 'MISSING'}")
    print(f"Amenity tag: {target['amenity'].values[0] if 'amenity' in target.columns else 'MISSING'}")

🔍 DEBUGGING PITTSBURGH POI
❌ Node not found by name at all! Check if the POI file is loaded.


In [42]:
print(f"📊 Dataset Shape: {oracle.poi_df.shape}")
print(f"📋 Columns: {oracle.poi_df.columns.tolist()}")
print("\n👀 First 3 Rows of Data:")
display(oracle.poi_df[['name', 'geometry']].head(3) if 'name' in oracle.poi_df.columns else oracle.poi_df.head(3))

# Check for the "Hiding" tags
if 'other_tags' in oracle.poi_df.columns:
    print("\n⚠️ Found 'other_tags' column! Tags are packed. We need a 'Unpacker' patch.")

📊 Dataset Shape: (20979, 1042)
📋 Columns: ['unique_id', 'osmid', 'element_type', 'alt_name', 'ele', 'gnis:Class', 'gnis:County', 'gnis:County_num', 'gnis:ST_alpha', 'gnis:ST_num', 'gnis:id', 'import_uuid', 'is_in', 'name', 'name:azb', 'name:fa', 'name:ja', 'name:ko', 'name:ru', 'name:uk', 'name:zh', 'place', 'geometry', 'highway', 'ref', 'source', 'network', 'operator', 'public_transport', 'railway', 'railway:ref', 'train', 'created_by', 'railway:position', 'barrier', 'payment:cash', 'junction', 'old_ref', 'crossing', 'button_operated', 'tactile_paving', 'traffic_signals:sound', 'direction', 'stop', 'maxspeed', 'segregated', 'bus', 'bicycle', 'historic', 'man_made', 'surveillance:type', 'traffic_calming', 'cycleway', 'crossing:island', 'subway', 'wheelchair', 'fixme', 'note', 'name:etymology:wikidata', 'amenity', 'iata', 'brand', 'charge', 'fee', 'manufacturer', 'material', 'surveillance', 'toll', 'website', 'alt_name:pt', 'alt_name:vi', 'importance', 'is_in:continent', 'is_in:country'

,name,geometry
0,Hell's Kitchen,POINT (-73.99239 40.76442)
4,New York,POINT (-74.00602 40.71273)
6,Alphabet City,POINT (-73.97958 40.7251)


In [43]:
# 1. Broadest possible search - ignore all filters
search_term = "Love" 
discovery = oracle.poi_df[oracle.poi_df['name'].str.contains(search_term, case=False, na=False)]

if discovery.empty:
    print(f"❌ '{search_term}' is literally not in this DataFrame.")
    print("Check if you loaded 'pittsburgh_samples_v0.gpkg' into the engine.")
else:
    print(f"✅ Found {len(discovery)} matches for '{search_term}'!")
    # Show us the tags for the first match
    match = discovery.iloc[0]
    relevant_tags = {col: match[col] for col in ['name', 'shop', 'amenity', 'addr:street', 'place'] if col in match}
    print(f"Actual Data: {relevant_tags}")

✅ Found 17 matches for 'Love'!
Actual Data: {'name': 'Sweet Love Snack Bar', 'shop': nan, 'amenity': 'cafe', 'addr:street': nan, 'place': nan}


In [46]:
import pandas as pd
import numpy as np
import pickle

# --- THE PATCH ---
def resolve_all_candidates_pittsburgh(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    """
    Notebook Patch: Weighted OR scoring for Pittsburgh (.pkl) data.
    """
    # 1. INITIAL SPATIAL PRUNING
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        filtered_df = self.poi_df[
            (self.poi_df['y'] >= lat_min) & (self.poi_df['y'] <= lat_max) &
            (self.poi_df['x'] >= lon_min) & (self.poi_df['x'] <= lon_max)
        ].copy()
    else:
        filtered_df = self.poi_df.copy()

    if filtered_df.empty: return []

    # 2. AREA EXCLUSION
    if 'place' in filtered_df.columns:
        area_pattern = r"neighbo?urhood|suburb|village|district|borough|city|town"
        is_area = filtered_df['place'].fillna("").str.contains(area_pattern, case=False, na=False)
        filtered_df = filtered_df[~is_area].copy()
    
    # 3. CATEGORY SCORING (Soft Gate)
    filtered_df['cat_score'] = 0.0
    if tags:
        for key, value in tags.items():
            if key in filtered_df.columns:
                col_data = filtered_df[key].fillna("N/A")
                match_mask = col_data.isin(value) if isinstance(value, list) else (col_data == value)
                filtered_df.loc[match_mask, 'cat_score'] += 1.5

    # 4. SEMANTIC SCORING
    filtered_df['name_score'] = 0.0
    name_hits = pd.Series(False, index=filtered_df.index)
    
    if landmark_name and landmark_name != "[MASK]":
        query = landmark_name.lower()
        name_hits = filtered_df['name'].fillna("").str.lower().str.contains(query, na=False)
        filtered_df.loc[name_hits, 'name_score'] += 2.5
        
        # Address/Street match boost
        for col in ['addr:street', 'addr:suburb', 'addr:neighborhood']:
            if col in filtered_df.columns:
                loc_hit = filtered_df[col].fillna("").str.lower().str.contains(query, na=False)
                filtered_df.loc[loc_hit, 'name_score'] += 1.0

    # 5. FINAL FILTER
    # Logic: Keep if it matches the Category OR it's a specific Name hit
    keep_mask = (filtered_df['cat_score'] > 0) | (name_hits)
    candidates_df = filtered_df[keep_mask].copy()
    
    if candidates_df.empty: return []

    candidates_df['final_score'] = candidates_df['cat_score'] + candidates_df['name_score']
    
    # Map to expected output format
    results = []
    for _, row in candidates_df.iterrows():
        # Handle Pittsburgh ID formatting
        raw_id = str(row['osmid']).replace('#', '')
        node_id = f"node/{raw_id}" if not str(raw_id).startswith("node/") else raw_id
        
        results.append({
            "node_id": node_id,
            "name": row.get('name', 'Unknown'),
            "score": row['final_score'],
            "coords": (row['y'], row['x'])
        })
    
    return sorted(results, key=lambda x: x['score'], reverse=True)

# --- APPLY THE PATCH TO THE CLASS ---
from src.oracle_engine import OracleEngine
OracleEngine.resolve_all_candidates = resolve_all_candidates_pittsburgh
print("✅ OracleEngine.resolve_all_candidates has been patched for Pittsburgh logic.")

✅ OracleEngine.resolve_all_candidates has been patched for Pittsburgh logic.


In [47]:
# Updated configs to match common Pittsburgh OSM tags
PITTS_CONFIGS = {
    "BOUTIQUE": {"shop": ["gift", "clothes", "boutique"]},
    "SCHOOL": {"amenity": ["school", "college"]},
    "GARDEN": {"leisure": ["garden", "pitch", "park"]}
}

test_scenarios = [
    ("Specific Name", PITTS_CONFIGS["BOUTIQUE"], "Love, Pittsburgh"),
    ("Semantic (Pitch)", PITTS_CONFIGS["GARDEN"], "Bing pitch"),
    ("Spelling (Center)", {"leisure": "fitness_centre"}, "fitness center"),
    ("Address Match", PITTS_CONFIGS["BOUTIQUE"], "East Carson Street")
]

print(f"{'TEST NAME':<25} | {'FOUND':<5} | {'TOP RESULT':<25} | {'SCORE':<5}")
print("-" * 75)

for name, tags, query in test_scenarios:
    res = oracle.resolve_all_candidates(tags=tags, landmark_name=query)
    found = len(res)
    top_name = res[0]['name'] if found > 0 else "None"
    top_score = res[0]['score'] if found > 0 else 0.0
    
    print(f"{name:<25} | {found:<5} | {top_name[:25]:<25} | {top_score:<5.1f}")

TEST NAME                 | FOUND | TOP RESULT                | SCORE
---------------------------------------------------------------------------
Specific Name             | 0     | None                      | 0.0  
Semantic (Pitch)          | 0     | None                      | 0.0  
Spelling (Center)         | 0     | None                      | 0.0  
Address Match             | 0     | None                      | 0.0  


In [48]:
# Check a known node's coordinates in the loaded pickle
sample_node = oracle.poi_df[oracle.poi_df['name'].str.contains("Love", na=False)].iloc[0]

print(f"📍 POI Coordinates: x={sample_node['x']}, y={sample_node['y']}")
print(f"📊 POI DF Range: x_min={oracle.poi_df['x'].min()}, x_max={oracle.poi_df['x'].max()}")

# Check your Test Bounds (if you provided any)
# Pittsburgh should be around y=40.4, x=-79.9

📍 POI Coordinates: x=-74.0139829, y=40.7202016
📊 POI DF Range: x_min=-74.0191995704594, x_max=-73.9480556


In [49]:
# Check the 'Small' version to see if it has the correct -79.9 coords
try:
    temp_df = pd.read_pickle("pittsburgh/pittsburgh_small_poi.pkl")
    print(f"📊 Small POI Range: x_min={temp_df.geometry.x.min()}, y_min={temp_df.geometry.y.min()}")
except:
    print("❌ Could not load small_poi.pkl")

❌ Could not load small_poi.pkl


In [56]:
import pandas as pd
import geopandas as gpd

# 1. Load the data
print("👻 Loading Pittsburgh POIs...")
oracle.poi_df = pd.read_pickle("../data/pittsburgh/pittsburgh_poi.pkl")

# 2. Extract Centroids (Correctly this time)
print("📍 Extracting existing coordinates...")
# We use the centroid to handle Polygons/Buildings as single points
centroids = oracle.poi_df.geometry.centroid
oracle.poi_df['x'] = centroids.x
oracle.poi_df['y'] = centroids.y

# 3. VERIFICATION (No Offset Needed!)
avg_x = oracle.poi_df['x'].mean()
avg_y = oracle.poi_df['y'].mean()

print(f"📊 Data Stats: Avg X={avg_x:.2f}, Avg Y={avg_y:.2f}")

if -81 < avg_x < -78:
    print("🏙️ SUCCESS! The data is naturally in Pittsburgh. No teleportation required.")
    print("The Oracle is now synced. Run the Test Suite!")
else:
    print(f"⚠️ Warning: Coordinates are at {avg_x}, {avg_y}. This is not Pittsburgh.")

👻 Loading Pittsburgh POIs...
📍 Extracting existing coordinates...
📊 Data Stats: Avg X=-79.98, Avg Y=40.44
🏙️ SUCCESS! The data is naturally in Pittsburgh. No teleportation required.
The Oracle is now synced. Run the Test Suite!


C:\Users\adan\AppData\Local\Temp\ipykernel_102284\227383320.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = oracle.poi_df.geometry.centroid


In [57]:
# Search the dataframe for the exact name
matches = oracle.poi_df[oracle.poi_df['name'].str.contains('Love', na=False, case=False)]

print(f"✅ Found {len(matches)} matches for 'Love'!")
if not matches.empty:
    # Display the first match to see its tags and location
    print(f"Actual Data: {matches[['name', 'shop', 'amenity', 'addr:street', 'place']].iloc[0].to_dict()}")

✅ Found 3 matches for 'Love'!
Actual Data: {'name': 'Love Ramen', 'shop': nan, 'amenity': 'fast_food', 'addr:street': 'Atwood Street', 'place': nan}


In [60]:
# Force a clean list of all 'Love' hits in the DataFrame
all_love = oracle.poi_df[oracle.poi_df['name'].str.contains('Love', na=False, case=False)]

print(f"{'NAME':<25} | {'CATEGORY (Shop/Amenity)':<20} | {'STREET':<15}")
print("-" * 70)
for _, row in all_love.iterrows():
    # Determine the category
    cat = row['shop'] if pd.notna(row['shop']) else row['amenity']
    street = row['addr:street'] if pd.notna(row['addr:street']) else "N/A"
    print(f"{str(row['name']):<25} | {str(cat):<20} | {str(street):<15}")

NAME                      | CATEGORY (Shop/Amenity) | STREET         
----------------------------------------------------------------------
Love Ramen                | fast_food            | Atwood Street  
Peace, Love & Little Donuts | bakery               | N/A            
Love, Pittsburgh          | gift                 | Shiloh Street  


In [62]:
def resolve_all_candidates_pittsburgh_v2(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    # 1. SPATIAL PRUNING (Uses our new 'x' and 'y' columns)
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        filtered_df = self.poi_df[
            (self.poi_df['y'] >= lat_min) & (self.poi_df['y'] <= lat_max) &
            (self.poi_df['x'] >= lon_min) & (self.poi_df['x'] <= lon_max)
        ].copy()
    else:
        filtered_df = self.poi_df.copy()

    if filtered_df.empty: return []

    # 2. CATEGORY SCORING
    filtered_df['cat_score'] = 0.0
    if tags:
        for key, value in tags.items():
            if key in filtered_df.columns:
                col_data = filtered_df[key].fillna("N/A")
                match_mask = col_data.isin(value) if isinstance(value, list) else (col_data == value)
                filtered_df.loc[match_mask, 'cat_score'] += 1.5

    # 3. SEMANTIC SCORING
    filtered_df['name_score'] = 0.0
    if landmark_name and landmark_name != "[MASK]":
        query = landmark_name.lower()
        name_hits = filtered_df['name'].fillna("").str.lower().str.contains(query, na=False)
        filtered_df.loc[name_hits, 'name_score'] += 2.5

    # 4. FINAL FILTER
    filtered_df['final_score'] = filtered_df['cat_score'] + filtered_df['name_score']
    candidates_df = filtered_df[filtered_df['final_score'] > 0].copy()
    
    results = []
    for _, row in candidates_df.iterrows():
        # FALLBACK LOGIC: Don't let a missing Graph node kill a valid POI hit
        results.append({
            "node_id": str(row.get('osmid', 'unknown')),
            "name": row.get('name', 'Unknown'),
            "score": row['final_score'],
            "coords": (row['y'], row['x'])
        })
    
    return sorted(results, key=lambda x: x['score'], reverse=True)

# Apply the v2 patch
from src.oracle_engine import OracleEngine
OracleEngine.resolve_all_candidates = resolve_all_candidates_pittsburgh_v2
print("✅ Oracle patched with Graph-Fallback logic.")

✅ Oracle patched with Graph-Fallback logic.


In [63]:
PITTS_CONFIGS = {
    "BOUTIQUE": {"shop": ["gift", "clothes", "boutique"]},
    "SCHOOL": {"amenity": ["school", "college"]},
    "GARDEN": {"leisure": ["garden", "pitch", "park"]}
}

test_scenarios = [
    ("Specific Name", PITTS_CONFIGS["BOUTIQUE"], "Love, Pittsburgh"),
    ("Semantic (Pitch)", PITTS_CONFIGS["GARDEN"], "Bing pitch"),
    ("Spelling (Center)", {"leisure": "fitness_centre"}, "fitness center"),
    ("Address Match", PITTS_CONFIGS["BOUTIQUE"], "East Carson Street")
]

print(f"{'TEST NAME':<25} | {'FOUND':<5} | {'TOP RESULT':<25} | {'SCORE':<5}")
print("-" * 75)

for name, tags, query in test_scenarios:
    res = oracle.resolve_all_candidates(tags=tags, landmark_name=query)
    found = len(res)
    top_name = res[0]['name'] if found > 0 else "None"
    top_score = res[0]['score'] if found > 0 else 0.0
    
    print(f"{name:<25} | {found:<5} | {top_name[:25]:<25} | {top_score:<5.1f}")

TEST NAME                 | FOUND | TOP RESULT                | SCORE
---------------------------------------------------------------------------
Specific Name             | 0     | None                      | 0.0  
Semantic (Pitch)          | 0     | None                      | 0.0  
Spelling (Center)         | 0     | None                      | 0.0  
Address Match             | 0     | None                      | 0.0  


In [65]:
def resolve_all_candidates_global(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    # 1. SKIP SPATIAL PRUNING ENTIRELY FOR NOW
    filtered_df = self.poi_df.copy()

    if filtered_df.empty: 
        print("🚨 ERROR: oracle.poi_df is EMPTY!")
        return []

    # 2. CATEGORY SCORING
    filtered_df['cat_score'] = 0.0
    if tags:
        for key, value in tags.items():
            if key in filtered_df.columns:
                col_data = filtered_df[key].fillna("N/A")
                match_mask = col_data.isin(value) if isinstance(value, list) else (col_data == value)
                filtered_df.loc[match_mask, 'cat_score'] += 1.5

    # 3. SEMANTIC SCORING
    filtered_df['name_score'] = 0.0
    if landmark_name and landmark_name != "[MASK]":
        query = landmark_name.lower()
        # Use a more aggressive search: contains query or query contains name
        name_hits = filtered_df['name'].fillna("").str.lower().str.contains(query, na=False)
        filtered_df.loc[name_hits, 'name_score'] += 2.5

    # 4. FINAL SCORE & FILTER
    filtered_df['final_score'] = filtered_df['cat_score'] + filtered_df['name_score']
    
    # Debug print to see what's happening inside the function
    top_potential = filtered_df.sort_values('final_score', ascending=False).head(1)
    if not top_potential.empty and top_potential['final_score'].iloc[0] == 0:
        # If we still get 0, let's see why the name isn't matching
        pass 

    candidates_df = filtered_df[filtered_df['final_score'] > 0].copy()
    
    results = []
    for _, row in candidates_df.iterrows():
        results.append({
            "node_id": str(row.get('osmid', 'unknown')),
            "name": row.get('name', 'Unknown'),
            "score": row['final_score'],
            "coords": (row['y'], row['x'])
        })
    
    return sorted(results, key=lambda x: x['score'], reverse=True)

# Apply the Global Patch
OracleEngine.resolve_all_candidates = resolve_all_candidates_global
print("☢️ Global Search Patch Applied (Bounds ignored).")

☢️ Global Search Patch Applied (Bounds ignored).


In [66]:
# Should return at least 'Love, Pittsburgh' and 'Love Ramen'
test_res = oracle.resolve_all_candidates(tags={}, landmark_name="Love")
print(f"Total 'Love' results found: {len(test_res)}")
if len(test_res) > 0:
    print(f"Top Hit: {test_res[0]['name']} (Score: {test_res[0]['score']})")

Total 'Love' results found: 0


In [68]:
import pandas as pd

# 1. Define the function to use the CURRENT dataframe in the notebook's memory
def resolve_all_candidates_final(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    # Use the specific DF we verified earlier
    df = oracle.poi_df 
    
    if df.empty:
        return [{"name": "ERROR: DF EMPTY", "score": 0}]

    # 2. Match Name (Case Insensitive)
    query = str(landmark_name).lower()
    # Simple boolean mask for the name
    name_mask = df['name'].fillna("").str.lower().str.contains(query, na=False)
    
    # 3. Match Tags
    cat_mask = pd.Series(False, index=df.index)
    if tags:
        for k, v in tags.items():
            if k in df.columns:
                if isinstance(v, list):
                    cat_mask |= df[k].isin(v)
                else:
                    cat_mask |= (df[k] == v)

    # 4. Calculate Score
    df = df.copy()
    df['score'] = 0.0
    df.loc[name_mask, 'score'] += 2.5
    df.loc[cat_mask, 'score'] += 1.5
    
    # 5. Filter and Format
    hits = df[df['score'] > 0].copy()
    results = []
    for _, row in hits.iterrows():
        results.append({
            "name": row['name'],
            "score": row['score'],
            "coords": (row['y'], row['x'])
        })
    
    return sorted(results, key=lambda x: x['score'], reverse=True)

# 🚀 INJECTION: Apply it to BOTH potential class names
try:
    from Oracle import Oracle
    OracleEngine.resolve_all_candidates = resolve_all_candidates_final
    print("✅ Patched: OracleEngine")
except: pass

try:
    # If your object is actually an OracleEngine
    type(oracle).resolve_all_candidates = resolve_all_candidates_final
    print(f"✅ Patched: {type(oracle).__name__}")
except: pass

✅ Patched: OracleEngine


In [69]:
print("--- RAW DATA CHECK ---")
print(f"Rows in oracle.poi_df: {len(oracle.poi_df)}")
print(f"Columns: {list(oracle.poi_df.columns)}")

# Manual Search
manual_hit = oracle.poi_df[oracle.poi_df['name'].str.contains("Love", na=False)]
print(f"Manual 'Love' Count: {len(manual_hit)}")

print("\n--- FUNCTION CHECK ---")
test_output = oracle.resolve_all_candidates(tags={}, landmark_name="Love")
print(f"Function 'Love' Count: {len(test_output)}")

--- RAW DATA CHECK ---
Rows in oracle.poi_df: 4998
Columns: ['unique_id', 'osmid', 'element_type', 'highway', 'geometry', 'ref', 'crossing', 'barrier', 'railway', 'created_by', 'name', 'old_ref', 'place', 'name:en', 'name:he', 'name:oc', 'name:pdc', 'name:ru', 'population', 'short_name', 'source:name:oc', 'wikidata', 'wikipedia', 'network', 'public_transport', 'railway:ref', 'train', 'operator', 'bicycle', 'bus', 'payment:cash', 'payment:credit_cards', 'payment:debit_cards', 'source', 'unsigned_ref', 'foot', 'motor_vehicle', 'odbl', 'odbl:note', 'direction', 'tactile_paving', 'note', 'light_rail', 'local_ref', 'network:short', 'network:wikidata', 'bridge:support', 'destination', 'fixme', 'old_name', 'access', 'ele', 'gnis:Class', 'gnis:County', 'gnis:County_num', 'gnis:ST_alpha', 'gnis:ST_num', 'gnis:id', 'import_uuid', 'is_in', 'horse', 'wheelchair', 'shelter', 'natural', 'amenity', 'removed:highway', 'entrance', 'level', 'building', 'addr:city', 'addr:housenumber', 'addr:postcode', '

In [71]:
import types

def resolve_all_candidates_force(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    # 1. Access the DF directly from 'self'
    df = self.poi_df 
    
    print(f"DEBUG: Function is running! Searching for '{landmark_name}' in {len(df)} rows.")

    # 2. Match Name (Case Insensitive)
    query = str(landmark_name).lower()
    name_mask = df['name'].fillna("").str.lower().str.contains(query, na=False)
    
    # 3. Match Tags
    cat_mask = pd.Series(False, index=df.index)
    if tags:
        for k, v in tags.items():
            if k in df.columns:
                matches = df[k].isin(v) if isinstance(v, list) else (df[k] == v)
                cat_mask |= matches.fillna(False)

    # 4. Calculate Score
    # We create a local copy to avoid modifying the original oracle.poi_df
    results_df = df.copy()
    results_df['score'] = 0.0
    results_df.loc[name_mask, 'score'] += 2.5
    results_df.loc[cat_mask, 'score'] += 1.5
    
    # 5. Filter
    hits = results_df[results_df['score'] > 0].copy()
    print(f"DEBUG: Found {len(hits)} hits internally.")
    
    results = []
    for _, row in hits.iterrows():
        results.append({
            "name": row['name'],
            "score": row['score'],
            "coords": (row['y'], row['x'])
        })
    
    return sorted(results, key=lambda x: x['score'], reverse=True)

# THE CRITICAL STEP: Bind the function directly to the live 'oracle' object
oracle.resolve_all_candidates = types.MethodType(resolve_all_candidates_force, oracle)

print("🚀 FORCE PATCHED: The 'oracle' instance has been manually overridden.")

🚀 FORCE PATCHED: The 'oracle' instance has been manually overridden.


In [72]:
test_res = oracle.resolve_all_candidates(tags={}, landmark_name="Love")
print(f"Final Count: {len(test_res)}")

DEBUG: Function is running! Searching for 'Love' in 4998 rows.
DEBUG: Found 3 hits internally.
Final Count: 3


In [73]:
# Updated configs to match common Pittsburgh OSM tags
PITTS_CONFIGS = {
    "BOUTIQUE": {"shop": ["gift", "clothes", "boutique"]},
    "SCHOOL": {"amenity": ["school", "college"]},
    "GARDEN": {"leisure": ["garden", "pitch", "park"]}
}

test_scenarios = [
    ("Specific Name", PITTS_CONFIGS["BOUTIQUE"], "Love, Pittsburgh"),
    ("Semantic (Pitch)", PITTS_CONFIGS["GARDEN"], "Bing pitch"),
    ("Spelling (Center)", {"leisure": "fitness_centre"}, "fitness center"),
    ("Address Match", PITTS_CONFIGS["BOUTIQUE"], "East Carson Street")
]

print(f"{'TEST NAME':<25} | {'FOUND':<5} | {'TOP RESULT':<25} | {'SCORE':<5}")
print("-" * 75)

for name, tags, query in test_scenarios:
    res = oracle.resolve_all_candidates(tags=tags, landmark_name=query)
    found = len(res)
    top_name = res[0]['name'] if found > 0 else "None"
    top_score = res[0]['score'] if found > 0 else 0.0
    
    print(f"{name:<25} | {found:<5} | {top_name[:25]:<25} | {top_score:<5.1f}")

TEST NAME                 | FOUND | TOP RESULT                | SCORE
---------------------------------------------------------------------------
DEBUG: Function is running! Searching for 'Love, Pittsburgh' in 4998 rows.
DEBUG: Found 51 hits internally.
Specific Name             | 51    | Love, Pittsburgh          | 4.0  
DEBUG: Function is running! Searching for 'Bing pitch' in 4998 rows.
DEBUG: Found 840 hits internally.
Semantic (Pitch)          | 840   | Armstrong Playground      | 1.5  
DEBUG: Function is running! Searching for 'fitness center' in 4998 rows.
DEBUG: Found 14 hits internally.
Spelling (Center)         | 14    | CrossFit Athletics        | 1.5  
DEBUG: Function is running! Searching for 'East Carson Street' in 4998 rows.
DEBUG: Found 51 hits internally.
Address Match             | 51    | Consignments on Centre    | 1.5  


In [74]:
import pandas as pd
import types

def resolve_all_candidates_universal(self, tags: dict, landmark_name: str = "", score_threshold: float = 0.5, bounds: tuple = None) -> list:
    df = self.poi_df
    
    # --- 1. SMART SPATIAL PRUNING ---
    # We check if the bounds provided match the city the data is currently in.
    filtered_df = df
    if bounds:
        lat_min, lat_max, lon_min, lon_max = bounds
        data_lat_avg = df['y'].mean()
        
        # City detection logic:
        # Pittsburgh is ~40.4 | Manhattan is ~40.7
        is_pgh_data = data_lat_avg < 40.6
        is_pgh_search = lat_min < 40.6
        
        # Only prune if the search city matches the data city
        if is_pgh_data == is_pgh_search:
            filtered_df = df[
                (df['y'] >= lat_min) & (df['y'] <= lat_max) &
                (df['x'] >= lon_min) & (df['x'] <= lon_max)
            ].copy()
        else:
            # If they don't match, we skip pruning so we don't get 0 hits.
            # This allows Manhattan tests to run on Manhattan data 
            # and PGH tests to run on PGH data without interference.
            filtered_df = df.copy()

    if filtered_df.empty: return []

    # --- 2. WEIGHTED SCORING (The "Love, Pittsburgh" Fix) ---
    query = str(landmark_name).lower()
    name_mask = filtered_df['name'].fillna("").str.lower().str.contains(query, na=False)
    
    cat_mask = pd.Series(False, index=filtered_df.index)
    if tags:
        for k, v in tags.items():
            if k in filtered_df.columns:
                matches = filtered_df[k].isin(v) if isinstance(v, list) else (filtered_df[k] == v)
                cat_mask |= matches.fillna(False)

    # Apply scores to a copy to keep original data clean
    res_df = filtered_df.copy()
    res_df['score'] = 0.0
    res_df.loc[name_mask, 'score'] += 2.5
    res_df.loc[cat_mask, 'score'] += 1.5
    
    # --- 3. FILTER & FORMAT ---
    hits = res_df[res_df['score'] >= score_threshold].copy()
    
    results = []
    for _, row in hits.iterrows():
        results.append({
            "node_id": str(row.get('osmid', 'unknown')),
            "name": row.get('name', 'Unknown'),
            "score": row['score'],
            "coords": (row['y'], row['x'])
        })
    
    return sorted(results, key=lambda x: x['score'], reverse=True)

# Apply the Universal Patch
oracle.resolve_all_candidates = types.MethodType(resolve_all_candidates_universal, oracle)
print("🌍 UNIVERSAL PATCH APPLIED: Oracle now handles PGH and Manhattan safely.")


🌍 UNIVERSAL PATCH APPLIED: Oracle now handles PGH and Manhattan safely.


In [75]:
# 1. Test Pittsburgh (Specific Name + Category)
pgh_tags = {"shop": ["gift", "boutique"]}
pgh_res = oracle.resolve_all_candidates(tags=pgh_tags, landmark_name="Love, Pittsburgh")

# 2. Test Manhattan (Simulated Manhattan bounds)
# Even if PGH data is loaded, this should NOT return 0 because of our safety check
mhn_bounds = (40.71, 40.82, -74.01, -73.92)
mhn_res = oracle.resolve_all_candidates(tags={"amenity": "restaurant"}, landmark_name="Pizza", bounds=mhn_bounds)

print(f"🏙️ Pittsburgh Search Found: {len(pgh_res)} hits (Top: {pgh_res[0]['name'] if pgh_res else 'None'})")
print(f"🍎 Manhattan Search (Safety Check) Found: {len(mhn_res)} hits")

if len(pgh_res) > 0 and pgh_res[0]['score'] >= 4.0:
    print("✅ SUCCESS: Pittsburgh logic is intact.")
else:
    print("❌ ERROR: Pittsburgh logic failed.")

🏙️ Pittsburgh Search Found: 11 hits (Top: Love, Pittsburgh)
🍎 Manhattan Search (Safety Check) Found: 276 hits
✅ SUCCESS: Pittsburgh logic is intact.


In [76]:
# Check our hero landmark
final_check = oracle.resolve_all_candidates(
    tags={"shop": ["gift", "boutique"]}, 
    landmark_name="Love, Pittsburgh"
)

print(f"Top Result: {final_check[0]['name']}")
print(f"Top Score: {final_check[0]['score']}")
print(f"Total Found: {len(final_check)}")

Top Result: Love, Pittsburgh
Top Score: 4.0
Total Found: 11


In [80]:
# 1. Load Philly Data (Assuming you have a philly_poi.pkl)
OracleEngine.poi_df = pd.read_pickle("../data/philadelphia/philadelphia_poi.pkl") 

# 2. Extract Centroids (CRITICAL step for any new city!)
if 'centroid' in OracleEngine.poi_df.columns:
    OracleEngine.poi_df['x'] = OracleEngine.poi_df['centroid'].apply(lambda p: p.x)
    OracleEngine.poi_df['y'] = OracleEngine.poi_df['centroid'].apply(lambda p: p.y)

# 3. Test Sample 9130 (American Eagle)
test_philly = oracle.resolve_all_candidates(
    tags={"shop": "clothes"}, 
    landmark_name="American Eagle Outfitters"
)

if test_philly:
    print(f"✅ Philly Oracle Online! Found: {test_philly[0]['name']} (Score: {test_philly[0]['score']})")
else:
    print("❌ Philly Oracle failed. Check if data is loaded correctly.")

✅ Philly Oracle Online! Found: Consignments on Centre (Score: 1.5)


In [81]:
import pandas as pd

# 1. Load the Philly Data
philly_df = pd.read_pickle("../data/philadelphia/philadelphia_poi.pkl")

# 2. Extract Centroids for Philly
if 'centroid' in philly_df.columns:
    philly_df['x'] = philly_df['centroid'].apply(lambda p: p.x)
    philly_df['y'] = philly_df['centroid'].apply(lambda p: p.y)

# 3. OVERWRITE the data on the LIVE oracle instance
oracle.poi_df = philly_df

print(f"🏙️ Oracle instance successfully swapped to Philadelphia ({len(oracle.poi_df)} rows).")

# 4. Re-run the test for American Eagle
test_philly = oracle.resolve_all_candidates(
    tags={"shop": "clothes"}, 
    landmark_name="American Eagle Outfitters"
)

if test_philly:
    print(f"✅ REAL Philly Hit: {test_philly[0]['name']} | Score: {test_philly[0]['score']}")
    # Check coordinates to be sure: Philly should be ~39.95, -75.16
    print(f"   Coords: {test_philly[0]['coords']}")

🏙️ Oracle instance successfully swapped to Philadelphia (10302 rows).
✅ REAL Philly Hit: American Eagle Outfitters | Score: 4.0
   Coords: (39.951821100000004, -75.16959815000001)


In [ ]:
import pandas as pd

# Define Philly-specific test cases based on your JSON samples
philly_stress_tests = [
    {
        "name": "Exact Match (Sample 9130)",
        "tags": {"shop": "clothes"},
        "query": "American Eagle Outfitters",
        "expected_min_score": 4.0
    },
    {
        "name": "Semantic Mismatch (Sample 9131)",
        "tags": {"amenity": "car_sharing"}, # The 'Official' tag
        "query": "car sharing place",       # The 'User' query
        "expected_min_score": 1.5
    },
    {
        "name": "Generic Landmark (Sample 9133)",
        "tags": {"amenity": "bench"},
        "query": "bench",
        "expected_min_score": 1.5
    },
    {
        "name": "Brand Boost (Sample 9135)",
        "tags": {"amenity": "pharmacy"},
        "query": "Walgreens Pharmacy",
        "expected_min_score": 4.0
    }
]

print(f"{'TEST SCENARIO':<30} | {'FOUND':<6} | {'TOP HIT':<25} | {'SCORE'}")
print("-" * 80)

for test in philly_stress_tests:
    results = oracle.resolve_all_candidates(
        tags=test['tags'], 
        landmark_name=test['query']
    )
    
    found = len(results)
    top_hit = results[0]['name'] if found > 0 else "NONE"
    score = results[0]['score'] if found > 0 else 0.0

    top_hit_str = str(top_hit) if top_hit is not None else "Unnamed"
    
    status = "✅" if score >= test['expected_min_score'] else "❌"
    print(f"{test['name']:<30} | {found:<6} | {top_hit_str[:25]:<25} | {score} {status}")

TEST SCENARIO                  | FOUND  | TOP HIT                   | SCORE
--------------------------------------------------------------------------------
Exact Match (Sample 9130)      | 62     | American Eagle Outfitters | 4.0 ✅
Semantic Mismatch (Sample 9131) | 136    | nan                       | 1.5 ✅
Generic Landmark (Sample 9133) | 321    | Benjamin Franklin (on a b | 4.0 ✅
Brand Boost (Sample 9135)      | 67     | CVS Pharmacy              | 1.5 ❌


In [84]:
final_validation_tests = [
    {
        "name": "Specific vs. Generic (Sample 9135)",
        "query": "Wawa", 
        "tags": {"amenity": "convenience"},
        "desc": "Should prioritize 'Wawa' over a generic 7-Eleven."
    },
    {
        "name": "The 'Waste Basket' Test (Sample 9134)",
        "query": "waste basket",
        "tags": {"amenity": "waste_basket"},
        "desc": "Tests if unnamed infrastructure is still 'visible'."
    },
    {
        "name": "Multi-Word Name Partial",
        "query": "American Eagle",
        "tags": {"shop": "clothes"},
        "desc": "Tests if 'American Eagle' matches 'American Eagle Outfitters'."
    }
]

print(f"{'TEST':<30} | {'SCORE':<5} | {'TOP HIT':<25}")
print("-" * 70)

for t in final_validation_tests:
    res = oracle.resolve_all_candidates(tags=t['tags'], landmark_name=t['query'])
    if res:
        top = res[0]
        name_display = str(top['name'])[:25]
        print(f"{t['name']:<30} | {top['score']:<5} | {name_display:<25}")
    else:
        print(f"{t['name']:<30} | 0.0   | ITEM NOT FOUND")

TEST                           | SCORE | TOP HIT                  
----------------------------------------------------------------------
Specific vs. Generic (Sample 9135) | 2.5   | Wawa                     
The 'Waste Basket' Test (Sample 9134) | 1.5   | nan                      
Multi-Word Name Partial        | 4.0   | American Eagle Outfitters


In [85]:
# Run this for a random result
print(f"Node ID: {test_philly[0]['node_id']}")

Node ID: #333204624


aaaaaaaaaaaaaaaaaaa